In [0]:
df = spark.table('workspace.csv.btc_usd_ytd')

# Convert to pandas 
pdf = df.toPandas()

# Change event_ to date
pdf['event_date'] = pdf['event_date'].dt.date

# Rename event_date, close_price_usd, market_cap_usd and drop volume_usd
pdf.rename(columns={'event_date': 'date'}, inplace=True)
pdf.rename(columns={'close_price_usd': 'price'}, inplace=True)
pdf.rename(columns={'market_cap_usd': 'market_cap'}, inplace=True)
pdf.drop(columns=['volume_usd'], inplace=True)

# add target column 
pdf['target'] = pdf['price'].shift(-1)
pdf.dropna(inplace=True)

pdf=pdf.sort_values(by='date')

display(pdf) 

date,price,market_cap,target
2026-01-01,88764.12624075128,1.748854230148607E12,89978.91920361611
2026-01-02,89978.91920361611,1.7726433898136357E12,90614.40684381907
2026-01-03,90614.40684381907,1.7969433514234443E12,91879.5772256957
2026-01-04,91879.5772256957,1.8096784419592795E12,93879.9479734475
2026-01-05,93879.9479734475,1.8349873097770288E12,93649.6319764084
2026-01-06,93649.6319764084,1.8749776435616355E12,91286.03314397692
2026-01-07,91286.03314397692,1.8704142778605935E12,91027.66616983686
2026-01-08,91027.66616983686,1.823249641248364E12,90551.71363174269
2026-01-09,90551.71363174269,1.8181277091763096E12,90420.29795170767
2026-01-10,90420.29795170767,1.808660912220566E12,90856.55910152226


In [0]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# Features and Target
features = ['price', 'market_cap']
x = pdf[features]
y = pdf['target']
  
# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    x,y,
    shuffle = False,
    test_size = 0.2,
)

# Train the model using the training sets
model = LinearRegression()
model.fit(X_train, y_train)

# Make predictions using the testing set
Predictions = model.predict(X_test)


In [0]:
import pandas as pd

x_test_reset = X_test.reset_index(drop=True)
y_test_reset = y_test.reset_index(drop=True)

date_col = pdf.loc[X_test.index, 'date'].reset_index(drop=True)

# Build results table 

results = pd.DataFrame({
    'date': date_col, 
    'price': x_test_reset['price'].round(2), 
    'Predicted_price': Predictions.round(2), 
})

# DIfference 
results['difference'] = (results['Predicted_price'] - results['price']).round(2)
results['% Error'] = ((results['difference']/results['price'])*100).round(2)




#Results 
display(results)



date,price,Predicted_price,difference,% Error
2026-08-31,78552.78,78365.52,-187.26,-0.24
2026-09-01,77416.44,77227.17,-189.27,-0.24
2026-09-02,77297.13,77125.55,-171.58,-0.22
2026-09-03,81264.7,81056.26,-208.44,-0.26
2026-09-04,79671.26,79420.51,-250.75,-0.31
2026-09-05,79821.7,79592.66,-229.04,-0.29
2026-09-06,80329.35,80093.16,-236.19,-0.29
2026-09-07,79093.46,78861.86,-231.6,-0.29
2026-09-08,78450.75,78243.37,-207.38,-0.26
2026-09-09,78283.06,78086.64,-196.42,-0.25
